# OrcaPath CNN training (Google Colab)

Before running this notebook, upload these two folders to `My Drive/OrcaPath/`:

- `Marine-Orca-Bioacoustics-Research-Study/` — this project folder, excluding `.venv`, `frontend/node_modules`, `frontend/dist`, and `data/raw_audio/orcasound`.
- `orcasound/` — the downloaded dataset folder containing `train.csv` and `train_audio_files/`.

In Colab, select **Runtime → Change runtime type → T4 GPU** before running the cells.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil

DRIVE_ROOT = Path('/content/drive/MyDrive/OrcaPath')
PROJECT_SOURCE = DRIVE_ROOT / 'Marine-Orca-Bioacoustics-Research-Study'
DATASET_SOURCE = DRIVE_ROOT / 'orcasound'
PROJECT = Path('/content/orcapath')

assert PROJECT_SOURCE.exists(), f'Missing project folder: {PROJECT_SOURCE}'
assert (DATASET_SOURCE / 'train.csv').exists(), f'Missing train.csv in {DATASET_SOURCE}'

shutil.rmtree(PROJECT, ignore_errors=True)
shutil.copytree(PROJECT_SOURCE, PROJECT, ignore=shutil.ignore_patterns('.venv', 'node_modules', 'dist', '__pycache__'))
shutil.copytree(DATASET_SOURCE, PROJECT / 'data' / 'raw_audio' / 'orcasound', dirs_exist_ok=True)
%cd /content/orcapath
print('Project and dataset copied to Colab local storage.')

In [ ]:
!pip -q install -r requirements_api.txt

import torch
assert torch.cuda.is_available(), 'GPU is not enabled. Use Runtime → Change runtime type → T4 GPU, then reconnect.'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# The available local subset has 606 recordings.
# Use 12 epochs for the first comparison. Increase later only if validation metrics improve.
!python -m src.models.train_cnn --allow-partial --epochs 12 --batch-size 64 --target-recall 0.80

In [ ]:
# Save the model and final held-out metrics back to Google Drive.
OUTPUT = DRIVE_ROOT / 'cnn_output'
OUTPUT.mkdir(exist_ok=True)
shutil.copy2(PROJECT / 'src' / 'models' / 'orca_cnn.pt', OUTPUT / 'orca_cnn.pt')
shutil.copy2(PROJECT / 'data' / 'processed' / 'cnn_training_metrics.json', OUTPUT / 'cnn_training_metrics.json')
print(f'Saved results to {OUTPUT}')

## Bring the model back

Download `orca_cnn.pt` and `cnn_training_metrics.json` from `My Drive/OrcaPath/cnn_output/`. Put them in your local project at `src/models/orca_cnn.pt` and `data/processed/cnn_training_metrics.json`.

Then start the local API with `ORCA_MODEL_BACKEND=cnn` to test the CNN in the upload app.